In [1]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
import os
import nested_pandas as npd
import pyarrow as pa
from progressbar import progressbar
import pyarrow.parquet as pq
import pyarrow.compute as pc

from astropy.coordinates import SkyCoord
import astropy.units as u

In [2]:
dirname = ['/media/snad/ztf_features/dr24-features_astra-clr_pretrained/embed_no_lc/' + \
          'ztfdr24_astra_embeddings/dataset/Norder=' +  str(i) +'/' for i in range(4, 8) ]

In [9]:
k = 3
flist = os.listdir(dirname[k])
flist_ = [os.listdir(dirname[k] + '/' + flist[j]) for j in range(len(flist))]

In [4]:
# Masha's chosen field
ra_n = 246.6812277
dec_n = 54.9451809
c1 = SkyCoord(ra=ra_n*u.deg, dec=dec_n*u.deg, frame='icrs')


In [5]:
len(flist)

4

In [6]:
[len(flist_[i]) for i in range(len(flist_))]

[378, 16, 52, 260]

In [7]:
data_list = []
for i in progressbar(range(len(flist))):
    for j in range(len(flist_[i])):

        data_temp = pd.read_parquet(dirname[k] + '/' + flist[i] + '/' + flist_[i][j])

        data_temp['ra'] = data_temp['matches'].apply(lambda x: np.mean(x['objra']))
        data_temp['dec'] = data_temp['matches'].apply(lambda x: np.mean(x['objdec']))

        mask_ra = data_temp['ra'].apply(lambda x: abs(x - ra_n) < 5)
        mask_dec = data_temp['dec'].apply(lambda x: abs(x - dec_n) < 5)
        mask = np.logical_and(mask_ra, mask_dec)

        if sum(mask) > 0:
            data_mask = data_temp[mask]
            dist_list = []
            
            for l in range(data_mask.shape[0]):
                c2 = SkyCoord(ra=data_mask.iloc[l]['ra']*u.deg, dec=data_mask.iloc[l]['dec']*u.deg, frame='icrs')
                dist = c1.separation(c2).deg
                dist_list.append(dist)

            data_mask['distance_masha'] = dist_list
            data_list.append(data_mask)

if len(data_list) > 0:
    data_temp2 = pd.concat(data_list, ignore_index=True)
    data_temp2.to_parquet('data/Norder=' + str(k) + '.parquet')

    order = np.argsort(data_temp2['distance_masha'].values)
    print(data_temp2.iloc[order[0]][['ra','dec', 'distance_masha']])

else:
    print('Nothing survived!')


100% (4 of 4) |##########################| Elapsed Time: 0:02:36 Time:  0:02:360:32


Nothing survived!


In [10]:
data_surv = pd.read_parquet('data/Norder=1.parquet')

In [13]:
index = np.argsort(data_surv['distance_masha'].values)

In [16]:
data_surv.iloc[index[0]]

koid                                                     795107100000041
matches                {'objra': [246.5809, 246.58087, 246.58089, 246...
embedding_beggining    [0.7656, 0.385, 3.295, -5.875, -0.3901, 0.2324...
embedding_middle       [0.6377, 0.1873, 3.516, -5.496, -0.972, 0.1794...
embedding_end          [-0.10657, 0.04782, 2.717, -6.137, -0.907, 0.8...
_healpix_29                                           714238069234827986
ra                                                            246.580887
dec                                                            54.855835
distance_masha                                                  0.106356
Name: 66881, dtype: object